# xAI: credenciales dinámicas de Vault

`./demo.sh` incluye estas etapas. El plugin viene incluido en la imagen `josemerchan/vault-xai:2.2.0-ent` publicada en Docker Hub; no requiere compilación ni una copia local de su repositorio. La management key entra por `XAI_MANAGEMENT_API_KEY`, se custodia en `verify/secrets/xai_mgmt` y configura `verify/xai`.

La aplicación utiliza **Client Credentials** de `chatbot-xai` para leer `xai/creds/lumen?ttl=300`. No lee `secrets/xai`. En una instalación vacía el rol tiene TTL 5 min, máximo 10 min y caducidad nativa; en una existente se conserva el rol configurado por el usuario y se comprueba que admite el TTL solicitado.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/xai.sh


## Configuración sin mostrar claves


In [ ]:
%%bash
source verify/scripts/common.sh
vault read -format=json xai/config | jq '.data|{api_endpoint,team_id}'
vault read -format=json xai/roles/lumen | jq '.data|{acls,ttl,max_ttl,set_native_expiry}'


## Identidad del agente `chatbot-xai`

Requiere las aplicaciones y usuarios de Verify del [notebook 6](6_Lumen_Chatbot_Vault_IAM.ipynb) (`02-verify-apps.sh`): la aplicación `vault-chatbot-xai` se crea a partir de la plantilla de `chatbot/research`. Registra el agente con RAR obligatorio y selecciona xAI como proveedor del modelo.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/configure-xai.sh


## Probar inferencia y caducidad real

La prueba emite una clave desechable de 60 s, llama a Grok, comprueba denegaciones y espera a su caducidad. Solo revoca su propio lease. No imprime la clave, no revoca por prefijo y no afecta las claves que esté utilizando Lumen.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/verify-xai.sh


El plugin actual no implementa renovación. Lumen reutiliza una clave mientras cubre el presupuesto de ejecución y solicita otra cuando quedan 210 s o menos. Vault revoca la anterior al expirar y xAI aplica su caducidad nativa. Ver [ciclo de vida y trazas](chatbot/README.md#xai-con-claves-dinámicas).
